# SQL reference 5: subqueries, CTEs and set operations

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-reference-05-subqueries-ctes-and-set-operations.ipynb)

Scalar, correlated, `IN` and `EXISTS` subqueries, derived tables, CTEs and how Spark executes them, recursive queries and their Spark 4.0 alternatives, and `UNION`/`INTERSECT`/`EXCEPT` with their `ALL` variants. Practise it in [SQL practice 5: subqueries, CTEs and set operations](sql-practice-05-subqueries-ctes-and-set-operations.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and three helpers: `show(query)` runs a query and prints its rows untruncated, `plan(query)` returns what `EXPLAIN` prints, and `expected_error()` prints the error class of an example that fails on purpose.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Contents

| section | entries |
| --- | --- |
| [Subqueries](#subqueries) | [Scalar subqueries](#scalar-subqueries), [`IN` and `EXISTS` subqueries](#in-and-exists-subqueries), [Derived tables](#derived-tables) |
| [Common table expressions](#common-table-expressions) | [`WITH` clauses](#with-clauses), [Recursive queries and their Spark alternatives](#recursive-queries-and-their-spark-alternatives) |
| [Set operations](#set-operations) | [`UNION`, `INTERSECT` and `EXCEPT`](#union-intersect-and-except), [Comparing two tables](#comparing-two-tables) |
| [Choosing a form](#choosing-a-form) | [Join, subquery or CTE?](#join-subquery-or-cte) |

## Subqueries

### Scalar subqueries

A subquery in parentheses that returns **one row and one column** can be used wherever a value is expected: in
`SELECT`, `WHERE`, `HAVING` or `ORDER BY`.

| kind | example | evaluated |
| --- | --- | --- |
| uncorrelated | `WHERE amount > (SELECT avg(amount) FROM orders)` | once |
| correlated | `WHERE amount > (SELECT avg(amount) FROM orders i WHERE i.customer_id = o.customer_id)` | logically once per outer row; Spark rewrites it as an aggregate plus a join |

**Notes:** a scalar subquery that returns more than one row is an error; one that returns no rows yields NULL. Spark
requires correlated scalar subqueries to be aggregated (so they provably return one row).

In [ ]:
show("SELECT order_id, amount FROM orders WHERE amount > (SELECT avg(amount) FROM orders) ORDER BY order_id")
show("""
SELECT o.order_id, o.customer_id, o.amount,
       (SELECT round(avg(i.amount), 2) FROM orders i WHERE i.customer_id = o.customer_id) AS customer_avg
FROM orders o
WHERE o.amount > (SELECT avg(i.amount) FROM orders i WHERE i.customer_id = o.customer_id)
ORDER BY o.order_id
""")

### `IN` and `EXISTS` subqueries

| predicate | TRUE when | NULL behaviour |
| --- | --- | --- |
| `x IN (SELECT col ...)` | `x` equals some returned value | NULL (not FALSE) when there is no match and the subquery returned a NULL |
| `x NOT IN (SELECT col ...)` | `x` equals none of them | **no rows at all** if the subquery returns any NULL |
| `EXISTS (SELECT ... correlated)` | the subquery returns at least one row | never NULL |
| `NOT EXISTS (...)` | it returns none | never NULL |

**Notes:** Spark plans `IN`/`EXISTS` as a left semi join and `NOT EXISTS` as a left anti join. `NOT IN` becomes a
"null-aware" anti join, which is slower; prefer `NOT EXISTS`. The `SELECT` list inside `EXISTS` is ignored, so
`SELECT 1` is conventional.

In [ ]:
show("""
SELECT name FROM customers
WHERE customer_id IN (SELECT o.customer_id FROM orders o JOIN order_items i USING (order_id)
                      JOIN products p USING (sku) WHERE p.category = 'Garden')
ORDER BY name
""")
show("""
SELECT p.sku, p.category FROM products p          -- products never part of a paid order
WHERE NOT EXISTS (SELECT 1 FROM order_items i JOIN orders o USING (order_id)
                  WHERE i.sku = p.sku AND o.status = 'paid')
ORDER BY p.sku
""")

### Derived tables

A subquery in `FROM` (a derived table) is a temporary result you can filter, join and aggregate again. It is the
standard way to filter on an alias, a window function or an aggregate of an aggregate.

**Notes:** Spark does not require an alias for a derived table, but most engines do (`FROM (...) AS t`); give one
anyway. When the same derived table is needed twice, or the nesting gets deep, switch to a CTE.

In [ ]:
show("""
SELECT region, avg(total) AS avg_customer_total
FROM (SELECT c.region, o.customer_id, sum(o.amount) AS total
      FROM orders o JOIN customers c USING (customer_id) GROUP BY c.region, o.customer_id) AS per_customer
GROUP BY region ORDER BY region NULLS LAST
""")

## Common table expressions

### `WITH` clauses

```sql
WITH name1 AS (query1),
     name2 AS (query2 that may reference name1)
SELECT ... FROM name2 ...
```

| property | detail |
| --- | --- |
| scope | visible to later CTEs in the same `WITH` and to the main query only |
| readability | names each step of a pipeline; read top to bottom |
| execution | Spark **inlines** CTEs like views: a CTE referenced twice is usually computed twice (cache a temp view or rely on reuse of exchanges when it is expensive). Postgres 12+ inlines unless `MATERIALIZED`; Snowflake and BigQuery decide per query |
| `WITH` inside subqueries | allowed |

**Use it for:** multi-step transformations, giving a complex filter a name, and reusing one intermediate result in
several places of the main query.

In [ ]:
show("""
WITH customer_totals AS (
    SELECT customer_id, sum(amount) AS total FROM orders WHERE status <> 'returned' GROUP BY customer_id
),
benchmark AS (
    SELECT avg(total) AS avg_total FROM customer_totals
)
SELECT c.name, t.total, round(b.avg_total, 2) AS avg_total
FROM customer_totals t CROSS JOIN benchmark b JOIN customers c USING (customer_id)
WHERE t.total > b.avg_total
ORDER BY t.total DESC
""")

### Recursive queries and their Spark alternatives

`WITH RECURSIVE` walks hierarchies (org charts, bills of materials) and generates series in Postgres, Snowflake,
BigQuery, DuckDB and Databricks. **Spark 4.0 does not support it** (it arrives in Spark 4.1). In Spark use:

| need | Spark approach |
| --- | --- |
| a series of numbers or dates | `explode(sequence(start, stop[, step]))` or `range(n)` |
| a hierarchy of known, small depth | a fixed number of self joins (one per level) |
| an arbitrary-depth hierarchy | iterate from Python: join level by level until no new rows appear, or use GraphFrames |

```sql
-- Postgres / Snowflake / Spark 4.1: management chain of any depth
WITH RECURSIVE chain AS (
    SELECT emp_id, name, manager_id, 0 AS depth FROM employees WHERE manager_id IS NULL
    UNION ALL
    SELECT e.emp_id, e.name, e.manager_id, c.depth + 1 FROM employees e JOIN chain c ON e.manager_id = c.emp_id
)
SELECT * FROM chain;
```

In [ ]:
show("SELECT explode(sequence(DATE'2026-01-01', DATE'2026-01-05')) AS day")
show("""
SELECT e.name, m1.name AS manager, m2.name AS managers_manager,
       CASE WHEN m1.emp_id IS NULL THEN 0 WHEN m2.emp_id IS NULL THEN 1 ELSE 2 END AS depth
FROM employees e
LEFT JOIN employees m1 ON m1.emp_id = e.manager_id
LEFT JOIN employees m2 ON m2.emp_id = m1.manager_id
ORDER BY depth, e.emp_id
""")

## Set operations

### `UNION`, `INTERSECT` and `EXCEPT`

```sql
query1 { UNION | INTERSECT | EXCEPT | MINUS } [ALL | DISTINCT] query2
```

| operator | returns | duplicates |
| --- | --- | --- |
| `UNION ALL` | every row of both | kept: no shuffle, the cheap default |
| `UNION` (= `UNION DISTINCT`) | rows in either | removed (a full de-duplication) |
| `INTERSECT [DISTINCT]` | rows in both | removed |
| `INTERSECT ALL` | rows in both, as many times as the smaller count | kept |
| `EXCEPT [DISTINCT]` / `MINUS` | rows in the first but not the second | removed |
| `EXCEPT ALL` | subtracts counts | kept |

**Rules:** both sides need the same number of columns with compatible types; columns are matched **by position**
and named after the first query; NULLs compare as equal (unlike `=`); `ORDER BY` and `LIMIT` apply to the whole
result. Precedence: `INTERSECT` binds tighter than `UNION`/`EXCEPT`; use parentheses.

In [ ]:
show("""
SELECT customer_id, 'signup' AS kind, CAST(signup_date AS TIMESTAMP) AS at FROM customers WHERE customer_id <= 2
UNION ALL
SELECT customer_id, 'order', order_ts FROM orders WHERE customer_id <= 2
ORDER BY customer_id, at
""")
show("""
SELECT customer_id FROM orders WHERE status = 'paid'
INTERSECT
SELECT customer_id FROM orders WHERE status = 'shipped'
ORDER BY customer_id
""")
show("SELECT sku FROM order_items EXCEPT SELECT sku FROM products ORDER BY sku")   # sold but missing from the catalogue
show("SELECT region FROM customers WHERE customer_id IN (1, 2, 4) EXCEPT ALL SELECT region FROM customers WHERE customer_id = 1")

### Comparing two tables

`EXCEPT` in both directions is the quickest full comparison of two tables with the same columns: it finds rows that
were added, removed or changed (a changed row appears on both sides). Use `EXCEPT ALL` when duplicate counts matter.

In [ ]:
show("""
WITH old AS (SELECT * FROM VALUES (1, 'a@x'), (2, 'b@x'), (3, 'c@x') AS t(id, email)),
     new AS (SELECT * FROM VALUES (1, 'a@x'), (2, 'b@new'), (4, 'd@x') AS t(id, email))
SELECT 'removed or changed' AS side, * FROM (SELECT * FROM old EXCEPT SELECT * FROM new)
UNION ALL
SELECT 'added or changed', * FROM (SELECT * FROM new EXCEPT SELECT * FROM old)
ORDER BY id, side
""")

## Choosing a form

### Join, subquery or CTE?

| question | prefer |
| --- | --- |
| "does a related row exist?" | `EXISTS` / semi join (no duplicates, NULL-safe) |
| "which rows have no related row?" | `NOT EXISTS` / anti join |
| "I need columns from the other table" | a join, at the right grain |
| "compare each row with an aggregate of its group" | a window function ([window functions](sql-reference-06-window-functions.ipynb)), or a correlated scalar subquery |
| "a multi-step transformation" | CTEs |
| "the same rows from two queries stacked" | `UNION ALL` |

Optimisers turn most of these forms into the same plan; choose the one that states the intent most clearly, then
check the plan when performance matters.

**Next:** practise these entries in [SQL practice 5: subqueries, CTEs and set operations](sql-practice-05-subqueries-ctes-and-set-operations.ipynb).